# Reto final — Clasificación desbalanceada y regularización en regresión

**Estudiantes:**  
**Juan David Tejedor Medina**  
**Miguel Guerardo Moreno Aveldaño**

## Objetivos

1. Crear un problema de clasificación multiclase desbalanceado.
2. Analizar por qué la exactitud (*accuracy*) no es suficiente cuando las clases no tienen la misma frecuencia.
3. Incorporar regularización L2 y Dropout a una red neuronal de regresión.
4. Comparar de manera reproducible el modelo original y el regularizado mediante curvas de aprendizaje y métricas de prueba.

El cuaderno fija semillas aleatorias, separa entrenamiento, validación y prueba, y ajusta los escaladores únicamente con el conjunto de entrenamiento para evitar fuga de información.


### Edición para portafolio

Trabajo académico recuperado de la especialización. Se conservan el código, las explicaciones y las atribuciones originales. Se retiraron las salidas y los metadatos de ejecución para facilitar su lectura y revisión. Las conclusiones conservadas pertenecen a la entrega original; los entrenamientos de Deep Learning no se repitieron al organizar este repositorio. Ver [procedencia y autoría](../../docs/PROCEDENCIA.md).


In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

import tensorflow as tf
from tensorflow.keras import regularizers

warnings.filterwarnings("ignore")
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.float_format = "{:.4f}".format

print("Entorno preparado correctamente")
print("TensorFlow:", tf.__version__)
print("Semilla:", SEED)


## 1. Crear un problema de clasificación desbalanceado

Un conjunto está **desbalanceado** cuando algunas clases tienen muchos más ejemplos que otras. Esto ocurre, por ejemplo, cuando los eventos de interés son poco frecuentes. Se generarán 3.000 observaciones, 10 características y 4 clases con proporciones aproximadas de 55 %, 25 %, 13 % y 7 %.


In [ ]:
X_imb, y_imb = make_classification(
    n_samples=3000,
    n_features=10,
    n_informative=7,
    n_redundant=2,
    n_classes=4,
    n_clusters_per_class=1,
    class_sep=1.5,
    flip_y=0.02,
    weights=[0.55, 0.25, 0.13, 0.07],
    random_state=SEED,
)

conteos = pd.Series(y_imb).value_counts().sort_index()
tabla_distribucion = pd.DataFrame({
    "Clase": conteos.index,
    "Cantidad": conteos.values,
    "Porcentaje": 100 * conteos.values / len(y_imb),
})

print("Forma de las características:", X_imb.shape)
print("Forma de las etiquetas:", y_imb.shape)
tabla_distribucion


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
barras = ax.bar(
    tabla_distribucion["Clase"].astype(str),
    tabla_distribucion["Cantidad"],
    color=["#4C78A8", "#72B7B2", "#F2CF5B", "#E45756"],
    edgecolor="black",
)
for barra, porcentaje in zip(barras, tabla_distribucion["Porcentaje"]):
    ax.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + 25,
        f"{porcentaje:.1f}%",
        ha="center",
        fontweight="bold",
    )
ax.set(xlabel="Clase", ylabel="Número de ejemplos", title="Distribución desbalanceada de las clases")
ax.set_ylim(0, tabla_distribucion["Cantidad"].max() * 1.14)
plt.tight_layout()
plt.show()


La clase 0 domina el conjunto, mientras que la clase 3 es minoritaria. Por ello, una predicción correcta de muchos ejemplos de la clase 0 puede elevar la exactitud global aunque el desempeño en la clase 3 sea deficiente.


## 2. ¿Sigue siendo suficiente el *accuracy*?

Los datos se dividen de forma estratificada para conservar aproximadamente la misma proporción de clases en entrenamiento, validación y prueba. El escalador se ajusta solo con entrenamiento.


In [ ]:
X_train_full_i, X_test_i, y_train_full_i, y_test_i = train_test_split(
    X_imb, y_imb, test_size=0.20, stratify=y_imb, random_state=SEED
)
X_train_i, X_val_i, y_train_i, y_val_i = train_test_split(
    X_train_full_i,
    y_train_full_i,
    test_size=0.20,
    stratify=y_train_full_i,
    random_state=SEED,
)

scaler_imb = StandardScaler()
X_train_i_scaled = scaler_imb.fit_transform(X_train_i)
X_val_i_scaled = scaler_imb.transform(X_val_i)
X_test_i_scaled = scaler_imb.transform(X_test_i)

tabla_particiones = pd.DataFrame({
    "Partición": ["Entrenamiento", "Validación", "Prueba"],
    "Ejemplos": [len(y_train_i), len(y_val_i), len(y_test_i)],
})
tabla_particiones


In [ ]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

classifier_imb = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(10,), name="entrada"),
    tf.keras.layers.Dense(32, activation="relu", name="oculta_1"),
    tf.keras.layers.Dense(16, activation="relu", name="oculta_2"),
    tf.keras.layers.Dense(4, activation="softmax", name="salida"),
], name="clasificador_desbalanceado")

classifier_imb.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)

classifier_imb.summary()


In [ ]:
callbacks_imb = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=20, restore_best_weights=True
    )
]

history_imb = classifier_imb.fit(
    X_train_i_scaled,
    y_train_i,
    validation_data=(X_val_i_scaled, y_val_i),
    epochs=250,
    batch_size=32,
    callbacks=callbacks_imb,
    verbose=0,
    shuffle=False,
)

epocas_clasificador = len(history_imb.history["loss"])
print(f"Entrenamiento terminado en {epocas_clasificador} épocas.")


In [ ]:
hist_clas = pd.DataFrame(history_imb.history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(hist_clas["loss"], label="Entrenamiento")
axes[0].plot(hist_clas["val_loss"], label="Validación")
axes[0].set(title="Pérdida del clasificador", xlabel="Época", ylabel="Entropía cruzada")
axes[0].legend()

axes[1].plot(hist_clas["accuracy"], label="Entrenamiento")
axes[1].plot(hist_clas["val_accuracy"], label="Validación")
axes[1].set(title="Exactitud del clasificador", xlabel="Época", ylabel="Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
test_loss_imb, test_acc_imb = classifier_imb.evaluate(
    X_test_i_scaled, y_test_i, verbose=0
)
pred_probs_imb = classifier_imb.predict(X_test_i_scaled, verbose=0)
pred_clases_imb = np.argmax(pred_probs_imb, axis=1)

# Línea base: predecir siempre la clase más frecuente del entrenamiento.
clase_mayoritaria = int(pd.Series(y_train_i).mode().iloc[0])
pred_baseline = np.full_like(y_test_i, clase_mayoritaria)

tabla_clasificacion = pd.DataFrame([
    {
        "Modelo": "Línea base mayoritaria",
        "Parámetros": 0,
        "Épocas": 0,
        "Pérdida prueba": np.nan,
        "Accuracy": accuracy_score(y_test_i, pred_baseline),
        "Balanced accuracy": balanced_accuracy_score(y_test_i, pred_baseline),
        "F1 macro": f1_score(y_test_i, pred_baseline, average="macro", zero_division=0),
        "F1 ponderado": f1_score(y_test_i, pred_baseline, average="weighted", zero_division=0),
    },
    {
        "Modelo": "Red neuronal",
        "Parámetros": classifier_imb.count_params(),
        "Épocas": epocas_clasificador,
        "Pérdida prueba": test_loss_imb,
        "Accuracy": accuracy_score(y_test_i, pred_clases_imb),
        "Balanced accuracy": balanced_accuracy_score(y_test_i, pred_clases_imb),
        "F1 macro": f1_score(y_test_i, pred_clases_imb, average="macro"),
        "F1 ponderado": f1_score(y_test_i, pred_clases_imb, average="weighted"),
    },
])

tabla_clasificacion


In [ ]:
reporte_clases = pd.DataFrame(
    classification_report(
        y_test_i,
        pred_clases_imb,
        labels=[0, 1, 2, 3],
        target_names=["Clase 0", "Clase 1", "Clase 2", "Clase 3"],
        output_dict=True,
        zero_division=0,
    )
).T

print("Reporte detallado de la red neuronal:")
reporte_clases


In [ ]:
cm_imb = confusion_matrix(y_test_i, pred_clases_imb)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_imb,
    display_labels=["Clase 0", "Clase 1", "Clase 2", "Clase 3"],
)
disp.plot(cmap="Purples", values_format="d")
plt.title("Matriz de confusión — red neuronal")
plt.grid(False)
plt.tight_layout()
plt.show()


In [ ]:
acc_base = tabla_clasificacion.loc[0, "Accuracy"]
f1_base = tabla_clasificacion.loc[0, "F1 macro"]
acc_red = tabla_clasificacion.loc[1, "Accuracy"]
f1_red = tabla_clasificacion.loc[1, "F1 macro"]
recalls = reporte_clases.loc[["Clase 0", "Clase 1", "Clase 2", "Clase 3"], "recall"]
clase_mejor_recall = recalls.idxmax()
clase_peor_recall = recalls.idxmin()

print("CONCLUSIÓN DEL PUNTO 2")
print("-" * 72)
print(
    f"La línea base que siempre predice la clase {clase_mayoritaria} logra un "
    f"accuracy de {acc_base:.4f}, pero su F1 macro es solo {f1_base:.4f}."
)
print(
    f"La red neuronal alcanza accuracy={acc_red:.4f}, F1 macro={f1_red:.4f} "
    f"y balanced accuracy={tabla_clasificacion.loc[1, 'Balanced accuracy']:.4f}."
)
print(
    f"El recall más alto corresponde a {clase_mejor_recall} ({recalls.max():.4f}) "
    f"y el más bajo a {clase_peor_recall} ({recalls.min():.4f})."
)
print(
    "Por tanto, el accuracy no es suficiente en datos desbalanceados: debe "
    "complementarse con F1 macro, balanced accuracy, recall por clase y la "
    "matriz de confusión, porque estas métricas dan el mismo peso a las clases "
    "minoritarias y revelan errores que el promedio global puede ocultar."
)


## 3. Agregar regularización L2 y Dropout al regresor

Se emplea el conjunto de diabetes de `scikit-learn`, que contiene 10 variables predictoras y una variable objetivo numérica. Se comparan dos redes con las mismas capas densas:

- **Original:** 64 → 32 → 16 → 1, activaciones ReLU y salida lineal.
- **Regularizada:** la misma arquitectura, L2 de 0.001 en las capas ocultas y Dropout de 0.20 después de las dos primeras capas.

L2 penaliza pesos grandes; Dropout desactiva aleatoriamente parte de las unidades durante el entrenamiento. Ambos mecanismos buscan reducir el sobreajuste.


In [ ]:
diabetes = load_diabetes()
X_reg = diabetes.data
y_reg = diabetes.target

X_train_full_r, X_test_r, y_train_full_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=SEED
)
X_train_r, X_val_r, y_train_r, y_val_r = train_test_split(
    X_train_full_r, y_train_full_r, test_size=0.20, random_state=SEED
)

scaler_reg = StandardScaler()
X_train_r_scaled = scaler_reg.fit_transform(X_train_r)
X_val_r_scaled = scaler_reg.transform(X_val_r)
X_test_r_scaled = scaler_reg.transform(X_test_r)

pd.DataFrame({
    "Partición": ["Entrenamiento", "Validación", "Prueba"],
    "Ejemplos": [len(y_train_r), len(y_val_r), len(y_test_r)],
})


In [ ]:
def construir_regresor(regularizado=False):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    l2 = regularizers.l2(0.001) if regularizado else None

    capas = [
        tf.keras.layers.Input(shape=(10,), name="entrada"),
        tf.keras.layers.Dense(64, activation="relu", kernel_regularizer=l2, name="oculta_1"),
    ]
    if regularizado:
        capas.append(tf.keras.layers.Dropout(0.20, name="dropout_1"))
    capas.append(
        tf.keras.layers.Dense(32, activation="relu", kernel_regularizer=l2, name="oculta_2")
    )
    if regularizado:
        capas.append(tf.keras.layers.Dropout(0.20, name="dropout_2"))
    capas.extend([
        tf.keras.layers.Dense(16, activation="relu", kernel_regularizer=l2, name="oculta_3"),
        tf.keras.layers.Dense(1, activation="linear", name="salida"),
    ])

    modelo = tf.keras.Sequential(
        capas,
        name="regresor_regularizado" if regularizado else "regresor_original",
    )
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss=tf.keras.losses.MeanSquaredError(),
        metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")],
    )
    return modelo


regressor_original = construir_regresor(regularizado=False)
regressor_original.summary()


In [ ]:
callbacks_original = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=30, restore_best_weights=True
    )
]
history_original = regressor_original.fit(
    X_train_r_scaled,
    y_train_r,
    validation_data=(X_val_r_scaled, y_val_r),
    epochs=500,
    batch_size=32,
    callbacks=callbacks_original,
    verbose=0,
    shuffle=False,
)
epocas_original = len(history_original.history["loss"])
print(f"Modelo original entrenado en {epocas_original} épocas.")


In [ ]:
regressor_regularizado = construir_regresor(regularizado=True)
regressor_regularizado.summary()

callbacks_regularizado = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=30, restore_best_weights=True
    )
]
history_regularizado = regressor_regularizado.fit(
    X_train_r_scaled,
    y_train_r,
    validation_data=(X_val_r_scaled, y_val_r),
    epochs=500,
    batch_size=32,
    callbacks=callbacks_regularizado,
    verbose=0,
    shuffle=False,
)
epocas_regularizado = len(history_regularizado.history["loss"])
print(f"Modelo regularizado entrenado en {epocas_regularizado} épocas.")


## 4. Comparar el modelo original contra el regularizado

Para ambos modelos se reportan arquitectura, parámetros entrenables, épocas efectivamente ejecutadas, MAE, RMSE y \(R^2\) sobre el mismo conjunto de prueba. MAE y RMSE deben minimizarse; \(R^2\) debe maximizarse.


In [ ]:
def calcular_metricas(y_real, y_pred):
    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2 = r2_score(y_real, y_pred)
    return mae, rmse, r2


y_pred_original = regressor_original.predict(X_test_r_scaled, verbose=0).ravel()
y_pred_regularizado = regressor_regularizado.predict(X_test_r_scaled, verbose=0).ravel()

mae_o, rmse_o, r2_o = calcular_metricas(y_test_r, y_pred_original)
mae_r, rmse_r, r2_r = calcular_metricas(y_test_r, y_pred_regularizado)

tabla_regresion = pd.DataFrame([
    {
        "Modelo": "Original",
        "Arquitectura": "10 → 64 → 32 → 16 → 1",
        "Regularización": "Ninguna",
        "Parámetros": regressor_original.count_params(),
        "Épocas": epocas_original,
        "MAE prueba": mae_o,
        "RMSE prueba": rmse_o,
        "R² prueba": r2_o,
    },
    {
        "Modelo": "Regularizado",
        "Arquitectura": "10 → 64 → 32 → 16 → 1",
        "Regularización": "L2=0.001 + Dropout=0.20",
        "Parámetros": regressor_regularizado.count_params(),
        "Épocas": epocas_regularizado,
        "MAE prueba": mae_r,
        "RMSE prueba": rmse_r,
        "R² prueba": r2_r,
    },
])

tabla_regresion


In [ ]:
hist_o = pd.DataFrame(history_original.history)
hist_r = pd.DataFrame(history_regularizado.history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(hist_o["loss"], label="Entrenamiento")
axes[0].plot(hist_o["val_loss"], label="Validación")
axes[0].set(title="Modelo original", xlabel="Época", ylabel="MSE")
axes[0].legend()

axes[1].plot(hist_r["loss"], label="Entrenamiento")
axes[1].plot(hist_r["val_loss"], label="Validación")
axes[1].set(title="Modelo regularizado", xlabel="Época", ylabel="MSE + penalización L2")
axes[1].legend()

fig.suptitle("Curvas de aprendizaje de los regresores", fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
residuos_original = y_test_r - y_pred_original
residuos_regularizado = y_test_r - y_pred_regularizado

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
axes[0].scatter(y_pred_original, residuos_original, alpha=0.70, color="#D85A30")
axes[0].axhline(0, linestyle="--", color="black")
axes[0].set(title="Residuos — modelo original", xlabel="Valor predicho", ylabel="Real − predicho")

axes[1].scatter(y_pred_regularizado, residuos_regularizado, alpha=0.70, color="#1D9E75")
axes[1].axhline(0, linestyle="--", color="black")
axes[1].set(title="Residuos — modelo regularizado", xlabel="Valor predicho")

plt.tight_layout()
plt.show()


In [ ]:
limites = [
    min(y_test_r.min(), y_pred_original.min(), y_pred_regularizado.min()),
    max(y_test_r.max(), y_pred_original.max(), y_pred_regularizado.max()),
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True, sharey=True)
for ax, pred, titulo, color in [
    (axes[0], y_pred_original, "Original", "#D85A30"),
    (axes[1], y_pred_regularizado, "Regularizado", "#1D9E75"),
]:
    ax.scatter(y_test_r, pred, alpha=0.70, color=color)
    ax.plot(limites, limites, "k--", label="Predicción perfecta")
    ax.set(title=titulo, xlabel="Valor real", ylabel="Valor predicho")
    ax.legend()

fig.suptitle("Valores reales frente a predicciones", fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
mejor_rmse = "Regularizado" if rmse_r < rmse_o else "Original"
mejor_r2 = "Regularizado" if r2_r > r2_o else "Original"
cambio_rmse = 100 * (rmse_r - rmse_o) / rmse_o
cambio_mae = 100 * (mae_r - mae_o) / mae_o

print("CONCLUSIÓN DE LA COMPARACIÓN DE REGRESIÓN")
print("-" * 72)
print(
    f"El modelo original obtuvo MAE={mae_o:.4f}, RMSE={rmse_o:.4f} y R²={r2_o:.4f}."
)
print(
    f"El modelo regularizado obtuvo MAE={mae_r:.4f}, RMSE={rmse_r:.4f} y R²={r2_r:.4f}."
)
print(
    f"Con L2 y Dropout, el MAE cambió {cambio_mae:+.2f}% y el RMSE {cambio_rmse:+.2f}% "
    "respecto al modelo original."
)

if rmse_r < rmse_o and r2_r > r2_o:
    print(
        "En esta ejecución la regularización ayudó a generalizar: redujo el error de "
        "prueba y aumentó R². Las curvas y los residuos respaldan esta comparación."
    )
elif rmse_r > rmse_o and r2_r < r2_o:
    print(
        "En esta ejecución la regularización no mejoró la generalización: aumentó el "
        "error de prueba y redujo R². Esto sugiere que L2=0.001 y Dropout=0.20 "
        "introdujeron demasiada restricción para este conjunto pequeño; regularizar no "
        "garantiza una mejora y sus hiperparámetros deben validarse."
    )
else:
    print(
        "Los indicadores presentan un resultado mixto. No puede afirmarse una mejora "
        "general sin priorizar una métrica y revisar conjuntamente curvas y residuos."
    )
print(f"El menor RMSE fue del modelo {mejor_rmse} y el mayor R² del modelo {mejor_r2}.")


## Conclusión general

Este reto mostró dos problemas diferentes de generalización:

1. En clasificación desbalanceada, la exactitud puede estar dominada por la clase mayoritaria. Por eso se debe complementar con F1 macro, *balanced accuracy*, recall por clase y matriz de confusión.
2. En regresión, L2 y Dropout modifican la complejidad efectiva de la red y pueden reducir el sobreajuste. Sin embargo, su utilidad se determina con el conjunto de validación y con métricas de prueba; no se debe asumir que siempre mejoran el resultado.

Las conclusiones numéricas anteriores se generan a partir de las salidas reales de esta ejecución y no de valores escritos manualmente.
